# Dropout

Wiki reference for [dropout](https://ml-viz-ruby.vercel.app/wiki/dropout).

**The idea in one sentence.** Dropout randomly zeroes activations during training (and **scales
up the survivors** so the expected value is unchanged — inverted dropout), which regularizes by
forcing redundancy — but it is a *cost*: it raises training loss, must be turned **off at
evaluation**, and too much of it underfits.

We implement inverted dropout from scratch and train a small net at several drop rates,
**validate that inverted dropout is unbiased and that dropout handicaps training**, then cover
the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
plt.style.use('dark_background')
torch.manual_seed(0)
rng = np.random.default_rng(0)

## 1 — Inverted dropout: verify E[h̃] = h

In [ ]:
def inverted_dropout(x, p, training=True):
    if not training or p == 0.0:
        return x
    mask = torch.bernoulli(torch.full_like(x, 1.0 - p))
    return x * mask / (1.0 - p)   # scale up survivors

# Verify E[inverted_dropout(x)] ≈ x
x = torch.tensor([1.0, 2.0, -3.0, 0.5])
p = 0.3
N = 50_000
samples = torch.stack([inverted_dropout(x, p) for _ in range(N)])
means = samples.mean(0)
print('Original x:         ', x.tolist())
print('E[dropout(x)]:      ', means.round(decimals=3).tolist())
print('Max deviation:      ', (means - x).abs().max().item())
print('\nAt test time (training=False), no mask is applied:')
print('output =', inverted_dropout(x, p, training=False).tolist())  # same as x

### Validate: inverted dropout is unbiased

Inverted dropout zeroes each unit with probability $p$ and **divides survivors by $1-p$**, so
the expected activation equals the original — which is why no rescaling is needed at test time.
We confirm $\mathbb{E}[\text{dropout}(x)] \approx x$ over many masks.

In [ ]:
print('x       :', x.tolist())
print('E[drop] :', means.round(decimals=3).tolist())
assert (means - x).abs().max() < 0.05, 'inverted dropout preserves the expected activation (unbiased)'
print('\n✅ scaling survivors by 1/(1-p) keeps E[dropout(x)] = x — no test-time rescaling needed')

## 2 — Ensemble interpretation: 2^n subnetworks

In [ ]:
# Show that dropout predictions approximate a large ensemble average
# We use a tiny 1-hidden-layer net on a regression task
torch.manual_seed(42)

class TinyNet(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.fc1 = nn.Linear(1, 64)
        self.fc2 = nn.Linear(64, 1)
        self.drop = nn.Dropout(p)

    def forward(self, x):
        return self.fc2(self.drop(F.relu(self.fc1(x))))

# Synthetic regression: y = sin(x) + noise
X_train = torch.linspace(-3, 3, 80).unsqueeze(1)
y_train = torch.sin(X_train) + 0.2 * torch.randn_like(X_train)

model = TinyNet(p=0.3)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for _ in range(1000):
    loss = F.mse_loss(model(X_train), y_train)
    opt.zero_grad(); loss.backward(); opt.step()

X_test = torch.linspace(-4, 4, 200).unsqueeze(1)

# MC dropout: run T forward passes with dropout active
model.train()   # keep dropout on
T = 100
with torch.no_grad():
    mc_preds = torch.stack([model(X_test) for _ in range(T)]).squeeze(-1)

mc_mean = mc_preds.mean(0).numpy()
mc_std  = mc_preds.std(0).numpy()
x_np = X_test.squeeze().numpy()

plt.figure(figsize=(10, 4))
plt.scatter(X_train.numpy(), y_train.numpy(), s=15, color='#6b7280', label='Training data', alpha=0.7)
plt.plot(x_np, np.sin(x_np), '--', color='#34d399', label='True sin(x)')
plt.plot(x_np, mc_mean, color='#6366f1', label='MC Dropout mean')
plt.fill_between(x_np, mc_mean - 2*mc_std, mc_mean + 2*mc_std, alpha=0.3, color='#6366f1',
                 label='±2σ (epistemic uncertainty)')
plt.axvspan(-3, 3, alpha=0.05, color='white', label='Training region')
plt.legend(fontsize=8); plt.xlabel('x'); plt.title('MC Dropout: uncertainty grows outside training region')
plt.tight_layout(); plt.show()

## 3 — Effect of drop rate on generalization

In [ ]:
from copy import deepcopy

def train_and_eval(p, n_epochs=800):
    torch.manual_seed(0)
    net = TinyNet(p=p)
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    for _ in range(n_epochs):
        net.train()
        F.mse_loss(net(X_train), y_train).backward()
        opt.step(); opt.zero_grad()
    net.eval()
    with torch.no_grad():
        train_loss = F.mse_loss(net(X_train), y_train).item()
        test_loss  = F.mse_loss(net(X_test), torch.sin(X_test)).item()
    return train_loss, test_loss

ps = [0.0, 0.2, 0.4, 0.6, 0.8]
results = [train_and_eval(p) for p in ps]
train_losses, test_losses = zip(*results)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ps, train_losses, 'o-', color='#6366f1', label='Train MSE')
ax.plot(ps, test_losses, 'o-', color='#f59e0b', label='Test MSE')
ax.set_xlabel('Drop rate p'); ax.set_ylabel('MSE loss')
ax.set_title('Train vs test loss as a function of dropout rate'); ax.legend()
plt.tight_layout(); plt.show()

### Validate: dropout is a regularizer with a cost — it handicaps training

Dropout makes each training step solve a harder (thinned) problem, so **training loss rises
monotonically with the drop rate**, and too much dropout ($p=0.8$) clearly **underfits**. (On
this small, smooth problem the extra regularization doesn't improve test loss — the benefit
appears when a model genuinely overfits.) We confirm the training-loss trend.

In [ ]:
print('p   :', ps)
print('train:', [round(t, 3) for t in train_losses])
print('test :', [round(t, 3) for t in test_losses])
assert all(train_losses[i] <= train_losses[i+1] + 1e-3 for i in range(len(train_losses)-1)), 'higher dropout raises training loss'
assert train_losses[-1] > 2 * train_losses[0], 'excessive dropout (p=0.8) underfits (train loss blows up)'
print('\n✅ dropout trades training fit for robustness — it is regularization, not free accuracy')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **forgetting model.eval()** | stochastic, wrong inference (demo) |
| **too much dropout** | underfitting — training loss blows up (verified) |
| **dropout on small/clean data** | no overfitting to fix → it only hurts (verified) |
| **placement** | dropout after activations; not usually on the output layer |
| **with batchnorm** | interacts awkwardly; order and rates need care |

Demo: train mode makes predictions stochastic; eval mode makes them deterministic.

In [ ]:
# The #1 dropout bug: forgetting model.eval(). In TRAIN mode dropout is active, so repeated
# forward passes on the SAME input give DIFFERENT (stochastic) outputs; you must switch to EVAL
# mode so dropout is disabled and predictions are deterministic. (That stochasticity is also
# exactly what MC-Dropout exploits to estimate uncertainty — but only on purpose.)
net = TinyNet(p=0.5)
net.train()
a1, a2 = net(X_test[:1]), net(X_test[:1])       # train mode: stochastic
net.eval()
with torch.no_grad():
    e1, e2 = net(X_test[:1]), net(X_test[:1])   # eval mode: deterministic
print(f'train-mode outputs differ: {not torch.allclose(a1, a2)}')
print(f'eval-mode outputs identical: {torch.allclose(e1, e2)}')
assert not torch.allclose(a1, a2), 'in train mode dropout makes predictions stochastic'
assert torch.allclose(e1, e2), 'model.eval() disables dropout -> deterministic predictions'
print('\nAlways call model.eval() before inference, or dropout corrupts your predictions.')

## ✏️ Your turn

**Task A — Spatial Dropout:** Implement a spatial dropout function for 4D tensors of shape `(N, C, H, W)` that drops entire channels. Apply it to a `(2, 8, 4, 4)` tensor with $p = 0.5$. Verify that each dropped channel is all zeros and each surviving channel is scaled by $1/(1-p)$.

**Task B — MC samples needed:** Run MC dropout with $T \in \{5, 10, 20, 50, 100, 200\}$ and compute the variance of the mean prediction across 20 different draws of $T$ samples. How many samples do you need before the estimate stabilizes?

In [ ]:
# Task A
def spatial_dropout(x, p, training=True):
    # x: (N, C, H, W)
    # TODO(you): generate a mask of shape (N, C, 1, 1) and apply with inverted scaling
    ...

x_4d = torch.ones(2, 8, 4, 4)
out_4d = spatial_dropout(x_4d, p=0.5)
# Each channel should be either all 0 or all (1 / 0.5 = 2.0)
if out_4d is not None:
    channel_means = out_4d.mean(dim=(-2,-1))
    print('Per-channel means (each should be 0 or 2.0):')
    print(channel_means)

# Task B
x_test_single = X_test[:1]
T_values = [5, 10, 20, 50, 100, 200]
model.train()
with torch.no_grad():
    mc_var_per_T = {}
    for T_ in T_values:
        means_across_draws = []
        for _ in range(20):
            preds = torch.stack([model(x_test_single) for _ in range(T_)]).squeeze()
            means_across_draws.append(preds.mean().item())
        mc_var_per_T[T_] = np.var(means_across_draws)
        print(f'T={T_:3d}: variance of mean estimate = {mc_var_per_T[T_]:.6f}')

<details><summary>Solution — Task A</summary>

```python
def spatial_dropout(x, p, training=True):
    if not training or p == 0.0:
        return x
    N, C, H, W = x.shape
    # Mask shape: (N, C, 1, 1) — broadcast over H, W
    mask = torch.bernoulli(torch.full((N, C, 1, 1), 1.0 - p))
    return x * mask / (1.0 - p)

x_4d = torch.ones(2, 8, 4, 4)
out = spatial_dropout(x_4d, p=0.5)
print(out[0, :, 0, 0])  # each channel: either 0.0 or 2.0
```
</details>

## Key takeaways

- **Inverted dropout is unbiased:** scaling survivors by $1/(1-p)$ keeps $\mathbb{E}[\cdot]=x$
  (verified) — no test-time rescaling.
- **Dropout is a regularizer with a cost:** it raises training loss and can underfit
  (verified) — the benefit appears when a model overfits.
- **Turn it off at eval:** `model.eval()` disables dropout; forgetting it corrupts predictions
  (demo).
- **MC-Dropout** turns that same stochasticity into an uncertainty estimate — on purpose.